# 006 Carga Imagenes

Flujo simplificado para preparar la carga al mosaico. No depende del Excel. La logica que antes estaba en `Parametros_ALL_v3.xlsx` queda declarada aqui como diccionarios y funciones: sector esperado, carpeta destino, fecha destino, nombre renombrado y manifiesto de copia.

## Parametros

- `FOLDER_INPUT`: carpeta entregada por el cliente; se busca recursivamente.
- `DESTINO_RAIZ`: raiz bajo la cual se crean las carpetas destino.
- `DESTINATION_FOLDER_BY_SECTOR`: diccionario editable de sector normalizado a carpeta destino.
- `DRY_RUN`: si esta en `True`, no copia; solo genera manifiesto.

In [ ]:
from datetime import datetime
from pathlib import Path
import importlib
import shutil

import pandas as pd

import core.mosaic_image_audit as mosaic_audit
mosaic_audit = importlib.reload(mosaic_audit)
from core.mosaic_image_audit import *

FOLDER_INPUT = Path(r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_Drone_Sin_Procesar\INPUT\20260519_Geosupport")
DESTINO_RAIZ = Path(r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO")

DRY_RUN = True
OVERWRITE = False

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path.cwd() / "outputs" / "carga_imagenes_006" / run_timestamp
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Modulo auditoria:", mosaic_audit.__file__)
print("Version logica:", AUDIT_LOGIC_VERSION)
print("Input:", FOLDER_INPUT)
print("Destino raiz:", DESTINO_RAIZ)
print("Salida:", OUTPUT_DIR)
print("DRY_RUN:", DRY_RUN)

## 1. Reglas internas de destino

Este diccionario reemplaza la dependencia del Excel. La llave es el `expected_sector` generado desde el nombre original; el valor es la carpeta destino bajo `DESTINO_RAIZ`.

Si un sector no esta en el diccionario queda en revision. Esto es intencional: evita copiar a una carpeta incorrecta cuando no existe una regla aprobada.

In [ ]:
# Reglas aprobadas / editables. Agregar o corregir aqui los destinos por sector.
DESTINATION_FOLDER_BY_SECTOR = {
    # Chacay / El Mauro
    "ed1": "Chacay_El_Mauro_Drone",
    "ed2": "Chacay_El_Mauro_Drone",
    "em2": "Chacay_El_Mauro_Drone",
    "em2_2": "Chacay_El_Mauro_Drone",
    "em3": "Chacay_El_Mauro_Drone",
    "ev1": "Chacay_El_Mauro_Drone",
    "subestacion_el_mauro": "Chacay_El_Mauro_Drone",

    # El Mauro
    "eb3": "El_Mauro_Drone",
    "ebd": "El_Mauro_Drone",
    "em4": "El_Mauro_Drone",
    "pozos_prp": "El_Mauro_Drone",
    "acceso_pozos_prp": "El_Mauro_Drone",
    "camino_alternativo_salamanca": "El_Mauro_Drone",

    # Chacay
    "em1": "Chacay_Drone",
    "estacion_cabecera": "Chacay_Drone",
    "estacion_cabeceras": "Chacay_Drone",

    # Puerto Punta Chungo / corredor
    "edt": "Puerto_Punta_Chungo_Drone",
    "ev2": "El_Mauro_Puerto_Punta_Chungo_Drone",

    # Reglas agregables segun revision operacional
    "ssee": "El_Mauro_Drone",
    "patio_19b": "Chacay_El_Mauro_Drone",
    "helipuerto_mauro": "El_Mauro_Drone",
    "helipuerto_mauro_mlp": "El_Mauro_Drone",
    "estacion_de_bombeo_no3": "El_Mauro_Drone",
    "estacion_intermedia_3": "Chacay_El_Mauro_Drone",
    "estacion_de_monitoreo_n_2": "El_Mauro_Drone",
    "torres_e48_a_e84": "Chacay_El_Mauro_Drone",
    "tramo_2_linea_33_kv_e_085_e_125": "Chacay_El_Mauro_Drone",
    "tramo_2_linea_33_kv_e_048_e_084": "Chacay_El_Mauro_Drone",
    "tramo_1_linea_33_kv_e_125_ml_eb2": "Chacay_El_Mauro_Drone",
    "tramo_2_linea_22_kv_e_35_e48": "Chacay_El_Mauro_Drone",
}

# Reglas de nombre especiales. Si una imagen genera un destino duplicado, se puede resolver aqui.
# Llave: file_name original. Valor: nombre destino con extension.
DESTINATION_FILE_NAME_OVERRIDES = {
    # Ejemplo:
    # "GEOSP-TRN-002636_GS_ORTOFOTO_HELIPUERTO MAURO_150526.tif": "CL_MLP_PAO_IF_Ortho_26_05_15_helipuerto_mauro_2.tif",
}

rules_df = pd.DataFrame(
    [{"expected_sector": sector, "destination_folder": folder} for sector, folder in DESTINATION_FOLDER_BY_SECTOR.items()]
)
display(rules_df.sort_values("expected_sector"))
rules_df.to_csv(OUTPUT_DIR / "01_destination_rules.csv", index=False, encoding="utf-8-sig")

## 2. Buscar imagenes TIF y construir nombre esperado

Se procesan `.tif` y `.tiff`. El nombre esperado se construye desde el nombre original usando las funciones del modulo `core.mosaic_image_audit`.

In [ ]:
input_images_df = scan_input_images(FOLDER_INPUT, extensions=ORTHO_MOSAIC_EXTENSIONS)
input_expected_df = add_expected_names(input_images_df)

print(f"Imagenes TIF/TIFF encontradas: {len(input_expected_df)}")
display(input_expected_df[[
    "file_name", "relative_path", "size_mb", "expected_name",
    "expected_date_token", "expected_sector", "rename_status", "date_warning"
]])

input_expected_df.to_csv(OUTPUT_DIR / "02_input_expected_names.csv", index=False, encoding="utf-8-sig")

## 3. Resolver destino

La ruta destino se construye asi:

`DESTINO_RAIZ / carpeta_por_sector / YY_MM_DD / nombre_renombrado.tif`

- `carpeta_por_sector`: sale de `DESTINATION_FOLDER_BY_SECTOR`.
- `YY_MM_DD`: sale del token de fecha detectado en el nombre original.
- `nombre_renombrado`: `expected_name + extension`, salvo override explicito.

In [ ]:
def build_destination_file_name(row):
    override = DESTINATION_FILE_NAME_OVERRIDES.get(row.get("file_name"))
    if override:
        return override

    expected_name = row.get("expected_name")
    extension = str(row.get("extension") or DEFAULT_RENAMED_EXTENSION).lower()
    return f"{expected_name}{extension}" if expected_name else None


def resolve_destination(row):
    if row.get("rename_status") != "ok":
        return {
            "destination_status": "review",
            "destination_rule": row.get("rename_status"),
            "destination_folder": None,
            "destination_date_folder": None,
            "destination_file_name": None,
            "destination_path": None,
            "review_reason": "No se pudo construir nombre esperado",
        }

    sector = row.get("expected_sector")
    folder = DESTINATION_FOLDER_BY_SECTOR.get(sector)
    date_folder = row.get("expected_date_token")
    destination_file_name = build_destination_file_name(row)

    if not folder:
        return {
            "destination_status": "review",
            "destination_rule": "missing_sector_rule",
            "destination_folder": None,
            "destination_date_folder": date_folder,
            "destination_file_name": destination_file_name,
            "destination_path": None,
            "review_reason": f"No existe regla para expected_sector={sector}",
        }

    destination_path = DESTINO_RAIZ / folder / date_folder / destination_file_name
    return {
        "destination_status": "ready",
        "destination_rule": "sector_dictionary",
        "destination_folder": folder,
        "destination_date_folder": date_folder,
        "destination_file_name": destination_file_name,
        "destination_path": str(destination_path),
        "review_reason": None,
    }


destination_df = pd.DataFrame([resolve_destination(row) for _, row in input_expected_df.iterrows()])
manifest_df = pd.concat([input_expected_df.reset_index(drop=True), destination_df], axis=1)

manifest_df["destination_exists"] = manifest_df["destination_path"].map(lambda value: Path(value).exists() if value else False)
manifest_df["duplicate_destination"] = manifest_df["destination_path"].notna() & manifest_df.duplicated("destination_path", keep=False)
manifest_df.loc[manifest_df["duplicate_destination"], "destination_status"] = "review"
manifest_df.loc[manifest_df["duplicate_destination"], "review_reason"] = "Destino duplicado dentro del manifiesto; usar DESTINATION_FILE_NAME_OVERRIDES"

copy_ready_df = manifest_df[manifest_df["destination_status"] == "ready"].copy()
review_df = manifest_df[manifest_df["destination_status"] != "ready"].copy()

summary_df = pd.DataFrame([
    {"metric": "run_timestamp", "value": run_timestamp},
    {"metric": "input_folder", "value": str(FOLDER_INPUT)},
    {"metric": "destination_root", "value": str(DESTINO_RAIZ)},
    {"metric": "input_tif_count", "value": len(input_expected_df)},
    {"metric": "copy_ready_count", "value": len(copy_ready_df)},
    {"metric": "review_count", "value": len(review_df)},
    {"metric": "destination_exists_count", "value": int(manifest_df["destination_exists"].sum())},
    {"metric": "duplicate_destination_count", "value": int(manifest_df["duplicate_destination"].sum())},
])

display(summary_df)
display(manifest_df.groupby(["destination_status", "destination_rule"], dropna=False).size().reset_index(name="count"))
display(manifest_df[["file_name", "expected_sector", "expected_name", "destination_status", "destination_folder", "destination_path", "review_reason"]])

summary_df.to_csv(OUTPUT_DIR / "03_summary.csv", index=False, encoding="utf-8-sig")
manifest_df.to_csv(OUTPUT_DIR / "04_copy_manifest.csv", index=False, encoding="utf-8-sig")
copy_ready_df.to_csv(OUTPUT_DIR / "05_copy_ready.csv", index=False, encoding="utf-8-sig")
review_df.to_csv(OUTPUT_DIR / "06_review_required.csv", index=False, encoding="utf-8-sig")

## 4. Copiar imagenes al destino

La celda respeta `DRY_RUN`. Para copiar realmente, revisar primero `04_copy_manifest.csv` y `06_review_required.csv`, resolver duplicados u omisiones, y cambiar `DRY_RUN = False`.

In [ ]:
copy_results = []

for _, row in copy_ready_df.iterrows():
    source_path = Path(row["path"])
    destination_path = Path(row["destination_path"])

    result = {
        "file_name": row["file_name"],
        "source_path": str(source_path),
        "destination_path": str(destination_path),
        "dry_run": DRY_RUN,
        "copied": False,
        "status": None,
        "message": None,
    }

    if not source_path.exists():
        result["status"] = "error"
        result["message"] = "No existe source_path"
    elif destination_path.exists() and not OVERWRITE:
        result["status"] = "skipped_exists"
        result["message"] = "Destino ya existe y OVERWRITE=False"
    elif DRY_RUN:
        result["status"] = "dry_run"
        result["message"] = "No se copio porque DRY_RUN=True"
    else:
        destination_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_path, destination_path)
        result["copied"] = True
        result["status"] = "copied"
        result["message"] = "Copiado correctamente"

    copy_results.append(result)

copy_results_df = pd.DataFrame(copy_results)
display(copy_results_df)
copy_results_df.to_csv(OUTPUT_DIR / "07_copy_results.csv", index=False, encoding="utf-8-sig")

print(f"Resultados exportados en: {OUTPUT_DIR}")